In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
data = {
    "Customer_ID": range(101, 111),
    "Age": [22, 25, 30, 35, 40, 45, 50, 55, 28, 38],
    "Income": [20000, 25000, 30000, 40000, 50000,
               60000, 70000, 80000, 28000, 45000],
    "Credit_Score": [500, 520, 550, 600, 650,
                     700, 750, 800, 530, 620],
    "Loan_Approved": [0, 0, 0, 0, 1, 1, 1, 1, 0, 1]
}

df = pd.DataFrame(data)

print("Original Dataset:")
display(df)

Original Dataset:


,Customer_ID,Age,Income,Credit_Score,Loan_Approved
0,101,22,20000,500,0
1,102,25,25000,520,0
2,103,30,30000,550,0
3,104,35,40000,600,0
4,105,40,50000,650,1
5,106,45,60000,700,1
6,107,50,70000,750,1
7,108,55,80000,800,1
8,109,28,28000,530,0
9,110,38,45000,620,1


In [3]:
print("Feature Leakage Example:")

df["Loan_Status_After_Decision"] = df["Loan_Approved"]

display(df[[
    "Age",
    "Income",
    "Credit_Score",
    "Loan_Approved",
    "Loan_Status_After_Decision"
]])

Feature Leakage Example:


,Age,Income,Credit_Score,Loan_Approved,Loan_Status_After_Decision
0,22,20000,500,0,0
1,25,25000,520,0,0
2,30,30000,550,0,0
3,35,40000,600,0,0
4,40,50000,650,1,1
5,45,60000,700,1,1
6,50,70000,750,1,1
7,55,80000,800,1,1
8,28,28000,530,0,0
9,38,45000,620,1,1


In [4]:
df["Approval_Information"] = df["Loan_Approved"]

print("Target Leakage Demonstration:")
display(df[[
    "Loan_Approved",
    "Approval_Information"
]])

print(
    "Correlation with Target:",
    df["Approval_Information"].corr(df["Loan_Approved"])
)

Target Leakage Demonstration:


,Loan_Approved,Approval_Information
0,0,0
1,0,0
2,0,0
3,0,0
4,1,1
5,1,1
6,1,1
7,1,1
8,0,0
9,1,1


Correlation with Target: 1.0


In [5]:
# Incorrect: Scaling the complete dataset before splitting

scaler = StandardScaler()

X = df[["Age", "Income", "Credit_Score"]]

X_scaled_incorrect = scaler.fit_transform(X)

print("Incorrect Scaling Completed:")
display(pd.DataFrame(
    X_scaled_incorrect,
    columns=X.columns
).head())

Incorrect Scaling Completed:


,Age,Income,Credit_Score
0,-1.431038,-1.295680,-1.255931
1,-1.140962,-1.034454,-1.050041
2,-0.657504,-0.773228,-0.741205
3,-0.174045,-0.250777,-0.226479
4,0.309414,0.271675,0.288247


In [6]:
temporal_data = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104, 105],
    "Transaction_Date": pd.to_datetime([
        "2024-01-01",
        "2024-02-01",
        "2024-03-01",
        "2024-04-01",
        "2024-05-01"
    ]),
    "Purchase_Amount": [1000, 2000, 1500, 3000, 2500],
    "Future_Purchase_Amount": [2000, 1500, 3000, 2500, 4000]
})

print("Temporal Leakage Example:")
display(temporal_data)

Temporal Leakage Example:


,Customer_ID,Transaction_Date,Purchase_Amount,Future_Purchase_Amount
0,101,2024-01-01,1000,2000
1,102,2024-02-01,2000,1500
2,103,2024-03-01,1500,3000
3,104,2024-04-01,3000,2500
4,105,2024-05-01,2500,4000


In [7]:
post_outcome_data = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104, 105],
    "Loan_Amount": [50000, 60000, 45000, 80000, 70000],
    "Loan_Approved": [1, 0, 1, 0, 1],
    "Recovery_Amount": [10000, 0, 15000, 0, 20000]
})

print("Post-Outcome Feature Example:")
display(post_outcome_data)

Post-Outcome Feature Example:


,Customer_ID,Loan_Amount,Loan_Approved,Recovery_Amount
0,101,50000,1,10000
1,102,60000,0,0
2,103,45000,1,15000
3,104,80000,0,0
4,105,70000,1,20000


In [8]:
transaction_data = pd.DataFrame({
    "Customer_ID": [101, 101, 102, 102, 103, 103],
    "Date": pd.to_datetime([
        "2024-01-01", "2024-03-01",
        "2024-01-05", "2024-04-01",
        "2024-01-10", "2024-05-01"
    ]),
    "Amount": [1000, 5000, 2000, 6000, 1500, 7000]
})

# Incorrect: includes future transactions
transaction_data["Total_Spending"] = (
    transaction_data.groupby("Customer_ID")["Amount"]
    .transform("sum")
)

display(transaction_data)

,Customer_ID,Date,Amount,Total_Spending
0,101,2024-01-01,1000,6000
1,101,2024-03-01,5000,6000
2,102,2024-01-05,2000,8000
3,102,2024-04-01,6000,8000
4,103,2024-01-10,1500,8500
5,103,2024-05-01,7000,8500


In [9]:
transaction_data = transaction_data.sort_values(
    ["Customer_ID", "Date"]
).copy()

transaction_data["Past_Spending"] = (
    transaction_data.groupby("Customer_ID")["Amount"]
    .transform(lambda x: x.cumsum() - x)
)

display(transaction_data)

,Customer_ID,Date,Amount,Total_Spending,Past_Spending
0,101,2024-01-01,1000,6000,0
1,101,2024-03-01,5000,6000,1000
2,102,2024-01-05,2000,8000,0
3,102,2024-04-01,6000,8000,2000
4,103,2024-01-10,1500,8500,0
5,103,2024-05-01,7000,8500,1500


In [10]:
encoding_data = pd.DataFrame({
    "City": ["Chennai", "Chennai", "Hyderabad",
             "Hyderabad", "Bangalore", "Bangalore"],
    "Loan_Approved": [1, 0, 1, 1, 0, 0]
})

# Incorrect: calculate encoding before splitting
encoding_data["City_Encoding_Incorrect"] = (
    encoding_data.groupby("City")["Loan_Approved"]
    .transform("mean")
)

display(encoding_data)

,City,Loan_Approved,City_Encoding_Incorrect
0,Chennai,1,0.5
1,Chennai,0,0.5
2,Hyderabad,1,1.0
3,Hyderabad,1,1.0
4,Bangalore,0,0.0
5,Bangalore,0,0.0


In [11]:
train_data = encoding_data.iloc[:4].copy()
test_data = encoding_data.iloc[4:].copy()

city_mapping = train_data.groupby("City")[
    "Loan_Approved"
].mean()

global_mean = train_data["Loan_Approved"].mean()

test_data["City_Encoding"] = (
    test_data["City"].map(city_mapping).fillna(global_mean)
)

print("Test Data Encoded Using Training Data Only:")
display(test_data)

Test Data Encoded Using Training Data Only:


,City,Loan_Approved,City_Encoding_Incorrect,City_Encoding
4,Bangalore,0,0.0,0.75
5,Bangalore,0,0.0,0.75


In [12]:
# Check suspicious features
print("Dataset Columns:")
print(df.columns.tolist())

print("\nCorrelation with Target:")
display(
    df.select_dtypes(include="number")
      .corr()["Loan_Approved"]
      .sort_values(ascending=False)
      .to_frame("Correlation")
)

Dataset Columns:
['Customer_ID', 'Age', 'Income', 'Credit_Score', 'Loan_Approved', 'Loan_Status_After_Decision', 'Approval_Information']

Correlation with Target:


,Correlation
Approval_Information,1.000000
Loan_Status_After_Decision,1.000000
Loan_Approved,1.000000
Age,0.850887
Income,0.846371
Credit_Score,0.844151
Customer_ID,0.591864


In [13]:
# Correct train-test split and scaling
X = df[["Age", "Income", "Credit_Score"]]
y = df["Loan_Approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data without fitting again
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)
print("Leakage-safe scaling completed!")

Training data shape: (8, 3)
Testing data shape: (2, 3)
Leakage-safe scaling completed!


In [14]:
# Create sample loan dataset
np.random.seed(42)

n = 500

leakage_demo = pd.DataFrame({
    "Age": np.random.randint(21, 60, n),
    "Income": np.random.randint(15000, 100000, n),
    "Credit_Score": np.random.randint(300, 850, n)
})

# Create target based on customer information
leakage_demo["Loan_Approved"] = (
    (leakage_demo["Income"] > 50000) &
    (leakage_demo["Credit_Score"] > 600)
).astype(int)

# Incorrect: this feature reveals the target
leakage_demo["Approval_Status_After_Decision"] = (
    leakage_demo["Loan_Approved"]
)

display(leakage_demo.head())

,Age,Income,Credit_Score,Loan_Approved,Approval_Status_After_Decision
0,59,99076,677,1,1
1,49,36545,350,0,0
2,35,36689,328,0,0
3,28,26174,761,0,0
4,41,24348,595,0,0


In [15]:
X_leak = leakage_demo[
    ["Age", "Income", "Credit_Score",
     "Approval_Status_After_Decision"]
]

y_leak = leakage_demo["Loan_Approved"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = (
    train_test_split(
        X_leak,
        y_leak,
        test_size=0.2,
        random_state=42,
        stratify=y_leak
    )
)

leak_model = LogisticRegression(max_iter=2000)

leak_model.fit(X_train_leak, y_train_leak)

leak_predictions = leak_model.predict(X_test_leak)

leak_accuracy = accuracy_score(
    y_test_leak,
    leak_predictions
)

print("Incorrect Model Accuracy:", round(leak_accuracy * 100, 2), "%")

Incorrect Model Accuracy: 100.0 %


In [16]:
X_correct = leakage_demo[
    ["Age", "Income", "Credit_Score"]
]

y_correct = leakage_demo["Loan_Approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X_correct,
    y_correct,
    test_size=0.2,
    random_state=42,
    stratify=y_correct
)

print("Training Features:")
display(X_train.head())

print("Testing Features:")
display(X_test.head())

Training Features:


,Age,Income,Credit_Score
40,46,40289,348
240,56,38954,776
282,24,70658,739
365,29,16917,414
281,39,99717,571


Testing Features:


,Age,Income,Credit_Score
193,36,96317,372
137,26,23702,577
333,48,28918,459
352,50,61514,337
71,45,45354,354


In [17]:
correct_model = LogisticRegression(max_iter=2000)

correct_model.fit(X_train, y_train)

correct_predictions = correct_model.predict(X_test)

correct_accuracy = accuracy_score(
    y_test,
    correct_predictions
)

print(
    "Correct Model Accuracy:",
    round(correct_accuracy * 100, 2),
    "%"
)

Correct Model Accuracy: 89.0 %


In [18]:
comparison = pd.DataFrame({
    "Model": [
        "Incorrect Model (With Leakage)",
        "Correct Model (Without Leakage)"
    ],
    "Accuracy": [
        leak_accuracy,
        correct_accuracy
    ]
})

comparison["Accuracy_Percentage"] = (
    comparison["Accuracy"] * 100
).round(2)

display(comparison)

,Model,Accuracy,Accuracy_Percentage
0,Incorrect Model (With Leakage),1.00,100.0
1,Correct Model (Without Leakage),0.89,89.0


In [19]:
leakage_summary = pd.DataFrame({
    "Topic": [
        "Feature Leakage",
        "Target Leakage",
        "Train-Test Leakage",
        "Temporal Leakage",
        "Post-Outcome Features",
        "Aggregation Leakage",
        "Target Encoding Leakage",
        "Detecting Leakage",
        "Preventing Leakage"
    ],
    "Purpose": [
        "Understand unavailable information entering a model",
        "Identify features revealing the target",
        "Prevent test information influencing training",
        "Avoid using future information",
        "Remove features created after outcomes",
        "Use only historical aggregation data",
        "Prevent target information contamination",
        "Identify suspicious model performance",
        "Use safe data splitting and preprocessing"
    ]
})

display(leakage_summary)

,Topic,Purpose
0,Feature Leakage,Understand unavailable information entering a ...
1,Target Leakage,Identify features revealing the target
2,Train-Test Leakage,Prevent test information influencing training
3,Temporal Leakage,Avoid using future information
4,Post-Outcome Features,Remove features created after outcomes
5,Aggregation Leakage,Use only historical aggregation data
6,Target Encoding Leakage,Prevent target information contamination
7,Detecting Leakage,Identify suspicious model performance
8,Preventing Leakage,Use safe data splitting and preprocessing
